Load the dataset

In [2]:
import pandas as pd
meta_qa = pd.read_parquet("parquet_data/qa_nl.parquet")
outp = pd.read_parquet("output/paraRAG/rag_outputs.parquet")
outp

,query_id,query,answer,retrieved_ids,latency_sec
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,"Een dagvaarding is een formeel, schriftelijk o...","[4788, 13580, 15046, 14700, 4826, 4953, 15044,...",23.495704
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Als één van de ouders alleen beslissingen moet...,"[3791, 3790, 3796, 3812, 3807, 3797, 3811, 385...",15.848987
2,305,Moet ik branddetectors installeren in Brussel?,Kort en duidelijk: ja — in het Brussels Hoofds...,"[23173, 2246, 23185, 2253, 2247, 2376, 23358, ...",22.450126
3,1811,"Als ik mijn consumentenkrediet niet betaal, ku...","Kort antwoord: ja, bij niet-betaling van een c...","[6112, 26822, 6120, 15740, 6083, 5963, 6135, 9...",26.167220
4,1558,Wat te doen als er een nieuwe huisgenoot in Br...,Kort antwoord: zorg dat bij de komst van een n...,"[4108, 2253, 4129, 4159, 4114, 1681, 2503, 373...",24.179190
...,...,...,...,...,...
1430,1106,De bal van mijn kind is in de tuin van mijn bu...,Ja. Volgens artikel 3.67 §1 van het Burgerlijk...,"[2715, 2710, 3795, 18, 23168, 3751, 5350, 5292...",31.783033
1431,134,Hoe lang moet men wachten voordat men een besl...,Kort antwoord: het bevoegde centrum (het OCMW/...,"[26673, 26676, 26678, 26674, 25977, 22026, 259...",27.087998
1432,1031,Ik woon samen met mijn partner en/of mijn kind...,Kort: bij een scheiding wordt de feitelijke do...,"[3791, 15007, 14608, 3797, 152, 5348, 3790, 25...",23.498632
1433,1715,Welke documenten en informatie mag het OCMW ei...,Kort antwoord: Het OCMW mag van de aanvrager a...,"[26674, 26673, 136, 26955, 1430, 1567, 8260, 6...",23.707709


In [3]:
#pip install deepeval

In [4]:
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from tqdm import tqdm

goldens = [Golden(input=question) for question in meta_qa.question]
dataset = EvaluationDataset(goldens=goldens)

for i, golden in enumerate(tqdm(dataset.goldens, desc="Building test cases")):
    test_case = LLMTestCase(
        input=meta_qa.question[i],
        actual_output=outp.answer[i],
        expected_output=meta_qa.answer[i]
    )
    dataset.add_test_case(test_case)

Building test cases: 100%|██████████| 1435/1435 [00:00<00:00, 4508.88it/s]


In [5]:
from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from deepeval.test_case import LLMTestCase, SingleTurnParams
correctness_metric = GEval(
    name="Correctness",
    criteria=(
        "Determine whether the actual_output is either an exact match to the "
        "expected_output, or a relevant statement that accurately conveys the "
        "same legal information."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    rubric=[
        Rubric(score_range=(0, 2), expected_outcome="Critical Failure/Incorrect — contradicts or misstates the law relative to the expected_output."),
        Rubric(score_range=(3, 4), expected_outcome="Poor/Significant Omissions — captures only a small fragment of the expected content or omits key legal elements."),
        Rubric(score_range=(5, 6), expected_outcome="Acceptable/Partially Complete — broadly correct but missing secondary details or nuance."),
        Rubric(score_range=(7, 9), expected_outcome="Good/Mostly Accurate — closely tracks the expected_output with only minor imprecision."),
        Rubric(score_range=(10, 10), expected_outcome="Excellent/Semantically Equivalent — fully and accurately conveys the same meaning as the expected_output, regardless of exact wording."),
    ],
    model = "gpt-5-mini-2025-08-07"
)

In [6]:
import math
import time
import pandas as pd
from tqdm import tqdm
from deepeval import evaluate
from deepeval.evaluate import DisplayConfig
from deepeval.evaluate import DisplayConfig, AsyncConfig

chunk_size = 100
max_concurrent = 20
max_retries = 3
base_delay = 2  # seconds, doubles each retry

test_cases = dataset.test_cases
n = len(test_cases)
num_chunks = math.ceil(n / chunk_size)

rows = []
failed_chunks = []

with tqdm(total=n, desc="Evaluating", unit="case") as pbar:
    for i in range(num_chunks):
        start = i * chunk_size
        chunk = test_cases[start:start + chunk_size]

        results = None
        for attempt in range(1, max_retries + 1):
            try:
                results = evaluate(
                    test_cases=chunk,
                    metrics=[correctness_metric],
                    display_config=DisplayConfig(show_indicator=True, print_results=False),
                    async_config=AsyncConfig(run_async=True, max_concurrent=max_concurrent),
                )
                break  # success, exit retry loop
            except Exception as e:
                if attempt == max_retries:
                    tqdm.write(f"Chunk {i} failed after {max_retries} attempts: {e}")
                    failed_chunks.append(i)
                else:
                    delay = base_delay * (2 ** (attempt - 1))
                    tqdm.write(f"Chunk {i} attempt {attempt} failed ({e}), retrying in {delay}s...")
                    time.sleep(delay)

        if results is not None:
            for test_result in results.test_results:
                if not test_result.metrics_data:
                    continue
                for metric in test_result.metrics_data:
                    rows.append({
                        "input": test_result.input,
                        "actual_output": test_result.actual_output,
                        "expected_output": test_result.expected_output,
                        "metric_name": metric.name,
                        "score": metric.score,
                        "reason": metric.reason,
                        "success": metric.success,
                    })

        pbar.update(len(chunk))

df = pd.DataFrame(rows)

Evaluating:   0%|          | 0/1435 [00:00<?, ?case/s]

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-5-mini-2025-08-07, strict=False, 
async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=851089;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 129.58s | token cost: 0.3164274999999998 USD)
» Test Results (100 total tests):
   » Pass Rate: 49.0% | Passed: 49 | Failed: 51

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

...

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-5-mini-2025-08-07, strict=False, 
async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=927783;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 56.66s | token cost: 0.10623424999999997 USD)
» Test Results (35 total tests):
   » Pass Rate: 42.86% | Passed: 15 | Failed: 20

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Evaluating: 100%|██████████| 1435/1435 [32:38<00:00,  1.36s/case]


In [7]:
# Save as Parquet
df.to_parquet("output/paraRAG/correctness_results.parquet", engine="pyarrow", index=False)

In [8]:
import pandas as pd
eval = pd.read_parquet("output/paraRAG/correctness_results.parquet")
eval

,input,actual_output,expected_output,metric_name,score,reason,success
0,Ik ben getrouwd. Moet ik een familiale aanspra...,Kort: op basis van de geraadpleegde wetsartike...,"U bent niet verplicht, maar het is aan te rade...",Correctness [GEval],0.6,The response correctly reaches the expected le...,True
1,Wat te doen als er een nieuwe huisgenoot in Br...,Kort antwoord: zorg dat bij de komst van een n...,Deze fiche is meer dan een jaar geleden bijgew...,Correctness [GEval],0.3,The Actual Output captures only a small fragme...,False
2,Wie oefent het ouderlijk gezag uit als één van...,Als één van de ouders alleen beslissingen moet...,Een ouder kan alleen bevoegd zijn om beslissin...,Correctness [GEval],0.4,The response accurately explains one important...,False
3,Ik word wettelijk samenwonend. We zijn ingesch...,Kort: Nee — simpelweg op hetzelfde adres inges...,Niet noodzakelijk. U wordt niet automatisch we...,Correctness [GEval],0.8,The Actual Output correctly answers the questi...,True
4,Ik heb een verklaring van wettelijke samenwoni...,Kort gezegd: wettelijke samenwoning regelt voo...,Er is geen enkel verschil tussen een kind dat ...,Correctness [GEval],0.7,The actual output correctly states that legal ...,True
...,...,...,...,...,...,...,...
1430,Welke documenten en informatie mag het OCMW ei...,Kort antwoord: Het OCMW mag van de aanvrager a...,U moet alle nuttige informatie meedelen zodat ...,Correctness [GEval],0.6,The Actual Output correctly addresses the same...,True
1431,Ik ben werkloos en heb kinderen. Heb ik recht ...,"Kort: ja, u kunt in bepaalde gevallen recht he...",Voordat u verdergaat: Alleen voormalige loontr...,Correctness [GEval],0.6,The response accurately conveys core legal rul...,True
1432,Hoe en door wie kan een traject terug naar wer...,"Kort antwoord: Een ""Terug Naar Werk""-traject w...","Voordat we verdergaan: let op, verwissel het t...",Correctness [GEval],0.4,The actual output overlaps with the expected o...,False
1433,Ik ben werkloos. Heb ik recht op moederschapsv...,Kort antwoord: als u volledig werkloos bent en...,Voordat we verder gaan: deze fiche bevat allee...,Correctness [GEval],0.1,Not equivalent. The Actual Output contradicts ...,False


In [9]:
#pip install ragas==0.4.3 langchain==0.3.30 langchain-community==0.3.31 langchain-openai==0.3.35

In [11]:
from retrievers import DocsRetriever
def get_context(article_ids):
    article_tuples = [(id_val, 0) for id_val in article_ids]
    return DocsRetriever(article_tuples).documents()
outp["retrieved_contexts"] = outp["retrieved_ids"].apply(get_context)

In [12]:
outp

,query_id,query,answer,retrieved_ids,latency_sec,retrieved_contexts
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,"Een dagvaarding is een formeel, schriftelijk o...","[4788, 13580, 15046, 14700, 4826, 4953, 15044,...",23.495704,[Art. 145.(Dagvaardingen wegens overtreding of...
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Als één van de ouders alleen beslissingen moet...,"[3791, 3790, 3796, 3812, 3807, 3797, 3811, 385...",15.848987,[Art. 374. (§ 1.) Wanneer de ouders niet samen...
2,305,Moet ik branddetectors installeren in Brussel?,Kort en duidelijk: ja — in het Brussels Hoofds...,"[23173, 2246, 23185, 2253, 2247, 2376, 23358, ...",22.450126,[Art. 4bis. Elke woning wordt uitgerust met mi...
3,1811,"Als ik mijn consumentenkrediet niet betaal, ku...","Kort antwoord: ja, bij niet-betaling van een c...","[6112, 26822, 6120, 15740, 6083, 5963, 6135, 9...",26.167220,[Art. VII.201 Onverminderd de andere gemeenrec...
4,1558,Wat te doen als er een nieuwe huisgenoot in Br...,Kort antwoord: zorg dat bij de komst van een n...,"[4108, 2253, 4129, 4159, 4114, 1681, 2503, 373...",24.179190,[Art. 1716_BRUSSELS_HOOFDSTEDELIJK_GEWEST. [NO...
...,...,...,...,...,...,...
1430,1106,De bal van mijn kind is in de tuin van mijn bu...,Ja. Volgens artikel 3.67 §1 van het Burgerlijk...,"[2715, 2710, 3795, 18, 23168, 3751, 5350, 5292...",31.783033,[Art. 3.67. Feitelijk gedogen van de eigenaar ...
1431,134,Hoe lang moet men wachten voordat men een besl...,Kort antwoord: het bevoegde centrum (het OCMW/...,"[26673, 26676, 26678, 26674, 25977, 22026, 259...",27.087998,[Art. 18. § 1. Het recht op maatschappelijke i...
1432,1031,Ik woon samen met mijn partner en/of mijn kind...,Kort: bij een scheiding wordt de feitelijke do...,"[3791, 15007, 14608, 3797, 152, 5348, 3790, 25...",23.498632,[Art. 374. (§ 1.) Wanneer de ouders niet samen...
1433,1715,Welke documenten en informatie mag het OCMW ei...,Kort antwoord: Het OCMW mag van de aanvrager a...,"[26674, 26673, 136, 26955, 1430, 1567, 8260, 6...",23.707709,[Art. 19.§ 1. Met het oog op de toekenning van...


In [13]:
import os
import asyncio
import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness
from tqdm.asyncio import tqdm_asyncio

load_dotenv()

client = AsyncOpenAI(api_key=os.getenv("OPENAI_API"))
llm = llm_factory("gpt-4o-mini", client=client, max_tokens=16384)
scorer = Faithfulness(llm=llm)

CONCURRENCY = 15
TIMEOUT = 90
MAX_RETRIES = 5
BATCH_SIZE = 100

async def score_row(i, row, sem):
    question = row["query"]
    answer = row["answer"]
    context = row["retrieved_contexts"]
    score = None

    async with sem:
        for attempt in range(MAX_RETRIES):
            try:
                result = await asyncio.wait_for(
                    scorer.ascore(
                        user_input=question,
                        response=answer,
                        retrieved_contexts=context,
                    ),
                    timeout=TIMEOUT,
                )
                score = result.value
                break
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(min(2 ** attempt, 20))
                else:
                    print(f"Row {i}: failed after {MAX_RETRIES} attempts ({e})")

    return {"question": question, "answer": answer, "faithfulness_score": score}

async def run_evaluation(outp: pd.DataFrame) -> pd.DataFrame:
    sem = asyncio.Semaphore(CONCURRENCY)
    all_results = [] 

    num_batches = (len(outp) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(num_batches):
        start = i * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(outp))
        batch_df = outp.iloc[start:end]

        tasks = [score_row(idx, row, sem) for idx, row in batch_df.iterrows()]
        batch_results = await tqdm_asyncio.gather(*tasks, desc=f"Batch {i + 1}/{num_batches}")
        all_results.extend(batch_results)

    results_df = pd.DataFrame(all_results)
    results_df.to_parquet("output/paraRAG/faithfulness_results.parquet", index=False)
    return results_df

# Usage:
results_df = await run_evaluation(outp)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Batch 15/15: 100%|██████████| 35/35 [01:00<00:00,  1.72s/it]
